# Exam 05 — LangChain & Agents

**Scope:** Course 4 — LCEL, Memory, Structured Output, Vector Stores, Advanced RAG, RAGAS, Tools, ReAct, LangGraph, LangSmith, Production Patterns  
**Format:** 20 MCQ (20 pts) · 15 SA (30 pts) · 10 Coding (30 pts) · 5 Case Studies (20 pts) = **100 pts**

---

In [1]:
import sys, types, numpy as np, re, json
from unittest.mock import MagicMock
for pkg in ["langchain","langchain_core","langchain_community","langchain_huggingface",
            "langgraph","langsmith","faiss","chromadb","ragas","pydantic"]:
    sys.modules.setdefault(pkg, types.ModuleType(pkg))

class FakeLLM:
    def __init__(self, replies=None):
        self._replies = replies or []; self._idx = 0
    def invoke(self, prompt):
        if self._replies:
            r = self._replies[self._idx % len(self._replies)]; self._idx += 1; return r
        return "mocked response"
    def stream(self, prompt):
        for tok in self.invoke(prompt).split(): yield tok + " "

class FakeEmbed:
    dim = 16
    def embed(self, text):
        v = np.zeros(self.dim)
        for i,w in enumerate(text.lower().split()): v[i%self.dim] += 1
        n = np.linalg.norm(v); return (v/n if n else v).tolist()

def cosine(a, b):
    a,b = np.array(a,float), np.array(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))

class FakeVectorStore:
    def __init__(self): self._docs=[]; self._vecs=[]; self._emb=FakeEmbed()
    def add(self, texts): [self._docs.append(t) or self._vecs.append(self._emb.embed(t)) for t in texts]
    def search(self, q, k=3):
        scores=[cosine(self._emb.embed(q),v) for v in self._vecs]
        top=sorted(range(len(scores)),key=lambda i:scores[i],reverse=True)[:k]
        return [self._docs[i] for i in top]

print("Mock OK")


Mock OK


## Part I — MCQ (20 × 1 pt)

### Q1. LCEL pipe

In LangChain ≥ 0.3, `prompt | llm | parser` creates:

- **A**: A Python generator object
- **B**: A `Runnable` (RunnableSequence) where each component's output feeds the next input
- **C**: A lazy iterator that must be materialised with `list()`
- **D**: A three-tuple of callables

In [2]:
q1 = ""

### Q2. stream vs invoke

`.stream()` on an LCEL chain vs `.invoke()`:

- **A**: Stream returns a list; invoke returns a generator
- **B**: Stream yields chunks incrementally as they are produced; invoke returns the complete result after generation finishes
- **C**: Both are identical
- **D**: Stream skips the output parser step

In [3]:
q2 = ""

### Q3. ConversationBufferMemory

ConversationBufferMemory stores:

- **A**: Only the most recent exchange
- **B**: The full conversation history as a list of message pairs
- **C**: A summary of the conversation
- **D**: Token counts only

In [4]:
q3 = ""

### Q4. RunnableWithMessageHistory

The key difference between raw `ConversationBufferMemory` and `RunnableWithMessageHistory` is:

- **A**: RunnableWithMessageHistory stores messages on disk
- **B**: RunnableWithMessageHistory integrates history injection into the LCEL chain so it is reusable and testable
- **C**: Raw memory is async; RunnableWithMessageHistory is sync
- **D**: RunnableWithMessageHistory only works with OpenAI

In [5]:
q4 = ""

### Q5. PydanticOutputParser

`PydanticOutputParser` validates LLM output by:

- **A**: Regex matching against field names
- **B**: Parsing the LLM's JSON string output and constructing a Pydantic model, raising if validation fails
- **C**: Asking the LLM to self-check its output
- **D**: Diffing against a schema file

In [6]:
q5 = ""

### Q6. with_structured_output

`llm.with_structured_output(Schema)` differs from `PydanticOutputParser` in that:

- **A**: It only works with Pydantic v1
- **B**: It uses the LLM provider's native function-calling/JSON mode to guarantee schema adherence at generation time
- **C**: It is slower
- **D**: It returns a raw string

In [7]:
q6 = ""

### Q7. MMR lambda

In MMR retrieval, the parameter lambda controls:

- **A**: The number of documents to retrieve
- **B**: The balance between relevance to the query (high lambda) and diversity among results (low lambda)
- **C**: The similarity threshold
- **D**: The embedding model temperature

In [8]:
q7 = ""

### Q8. multi-query retrieval

Multi-query retrieval generates multiple rewritten queries because:

- **A**: The embedding model is non-deterministic
- **B**: A single query may miss relevant documents due to vocabulary mismatch; multiple phrasings increase recall
- **C**: It reduces hallucination
- **D**: It is required by ChromaDB

In [9]:
q8 = ""

### Q9. RAGAS faithfulness

RAGAS Faithfulness score of 0.6 means:

- **A**: 60% of context was retrieved correctly
- **B**: 60% of the answer's atomic claims are grounded in the retrieved context
- **C**: The model answered 60% of questions correctly
- **D**: Context recall is 60%

In [10]:
q9 = ""

### Q10. LangGraph StateGraph

LangGraph `StateGraph` vs LCEL chain:

- **A**: StateGraph is strictly linear; LCEL supports cycles
- **B**: StateGraph supports cycles and conditional branching with shared mutable state; LCEL is a DAG
- **C**: They are identical
- **D**: StateGraph is deprecated

In [11]:
q10 = ""

### Q11. LangGraph END

In LangGraph, routing to `END` means:

- **A**: The graph raises StopIteration
- **B**: The workflow terminates and the final state is returned
- **C**: The current node is reset
- **D**: A cleanup node is called automatically

In [12]:
q11 = ""

### Q12. ReAct Action Input

In the manual ReAct loop in `03_agentflow`, after parsing `Action: web_search\nAction Input: LoRA rank`, the agent:

- **A**: Calls the LLM again immediately
- **B**: Dispatches `web_search` tool with argument `LoRA rank`, appends `Observation: <result>` to history
- **C**: Returns Final Answer
- **D**: Resets the history string

In [13]:
q12 = ""

### Q13. bind_tools error

`ValueError: This function requires a bind_tools() method` is raised when using `create_tool_calling_agent` with:

- **A**: Any encoder model
- **B**: `HuggingFacePipeline` which is text-generation only and has no native tool-schema API
- **C**: OpenAI models
- **D**: Models with temperature=0

In [14]:
q13 = ""

### Q14. LangSmith offline

In `course4/chapter4/class1_langsmith_tracing`, if `LANGSMITH_API_KEY` is not set, the tracer:

- **A**: Raises ImportError
- **B**: Falls back to writing traces as JSONL to a local file
- **C**: Disables tracing silently
- **D**: Prompts the user for a key

In [15]:
q14 = ""

### Q15. semantic cache purpose

The semantic cache in `class2_production_patterns` avoids:

- **A**: Duplicate documents in FAISS
- **B**: Redundant LLM inference calls for semantically equivalent queries
- **C**: Tokenisation overhead
- **D**: Memory fragmentation

In [16]:
q15 = ""

### Q16. FAISS vs ChromaDB persist

FAISS requires extra work for persistence because:

- **A**: FAISS uses SQL storage
- **B**: FAISS is in-memory only; you must call `faiss.write_index()` to serialise and `faiss.read_index()` to reload
- **C**: FAISS doesn't support serialisation
- **D**: ChromaDB also requires manual persistence

In [17]:
q16 = ""

### Q17. contextual compression

Contextual compression retrieval post-processes retrieved documents by:

- **A**: Re-ranking with a cross-encoder
- **B**: Extracting only the portions of each document relevant to the query, reducing noise
- **C**: Expanding each document with additional context
- **D**: Deduplicating identical passages

In [18]:
q17 = ""

### Q18. LangGraph human-in-loop

Human-in-the-loop in LangGraph is implemented by:

- **A**: Pausing the graph with `time.sleep()` until input arrives
- **B**: Using `interrupt_before` or `interrupt_after` to pause execution at a node, allowing external input before resuming
- **C**: Adding a special `human` node that always blocks
- **D**: Calling `graph.pause()` explicitly

In [19]:
q18 = ""

### Q19. return_full_text=False

Without `return_full_text=False` in `HuggingFacePipeline`, the LCEL chain returns:

- **A**: Only generated tokens
- **B**: The full prompt concatenated with generated tokens — the prompt is echoed in the output
- **C**: A structured dict
- **D**: An empty string

In [20]:
q19 = ""

### Q20. PromptTemplate variable

A `PromptTemplate` with `template='Answer: {question}'` called with `invoke({'question': 'What is LoRA?'})` produces:

- **A**: A raw string
- **B**: A formatted `HumanMessage` or `StringPromptValue` passed to the next chain component
- **C**: A Pydantic model
- **D**: A list of tokens

In [21]:
q20 = ""

### Auto-grader

In [22]:
KEY={"q1": "B", "q2": "B", "q3": "B", "q4": "B", "q5": "B", "q6": "B", "q7": "B", "q8": "B", "q9": "B", "q10": "B", "q11": "B", "q12": "B", "q13": "B", "q14": "B", "q15": "B", "q16": "B", "q17": "B", "q18": "B", "q19": "B", "q20": "B"}
answers={k:eval(k) for k in KEY}
correct=sum(1 for k,v in KEY.items() if answers[k]==v)
print(f'Part I: {correct}/{len(KEY)}')
for k,v in KEY.items():
    m='✓' if answers[k]==v else f'✗ ({v})'
    print(f'  {k}: {answers[k]}  {m}')

Part I: 0/20
  q1:   ✗ (B)
  q2:   ✗ (B)
  q3:   ✗ (B)
  q4:   ✗ (B)
  q5:   ✗ (B)
  q6:   ✗ (B)
  q7:   ✗ (B)
  q8:   ✗ (B)
  q9:   ✗ (B)
  q10:   ✗ (B)
  q11:   ✗ (B)
  q12:   ✗ (B)
  q13:   ✗ (B)
  q14:   ✗ (B)
  q15:   ✗ (B)
  q16:   ✗ (B)
  q17:   ✗ (B)
  q18:   ✗ (B)
  q19:   ✗ (B)
  q20:   ✗ (B)


---
## Part II — SA (15 × 2 pts)

### SA01. LCEL composition

Describe how `PromptTemplate | llm | StrOutputParser()` executes. What type does each `|` produce?

*Your answer here.*

### SA02. Memory types

Compare ConversationBufferMemory vs ConversationSummaryMemory. When would you use each?

*Your answer here.*

### SA03. Structured output reliability

Why is `llm.with_structured_output(Schema)` more reliable than `PydanticOutputParser` for getting schema-conformant output?

*Your answer here.*

### SA04. FAISS IndexHNSW vs Flat

When would you choose FAISS IndexHNSWFlat over IndexFlatL2 for a production RAG system?

*Your answer here.*

### SA05. RAG context window

Explain how chunk size and overlap affect RAG performance. What is the tradeoff?

*Your answer here.*

### SA06. RAGAS eval protocol

Describe the RAGAS evaluation workflow for a RAG pipeline using a golden set of 50 query-context-answer triples.

*Your answer here.*

### SA07. LangGraph cycles

Explain with a concrete example why an LCEL chain cannot implement a retry loop but LangGraph can.

*Your answer here.*

### SA08. Tool registry pattern

Describe the `ToolRegistry` pattern in `03_agentflow/src/graph.py`. How does it decouple tool registration from tool dispatch?

*Your answer here.*

### SA09. LangSmith trace anatomy

Describe the structure of a LangSmith trace for a single RAG query. What spans are created?

*Your answer here.*

### SA10. BM25 hybrid

How does hybrid BM25+dense retrieval work? When does BM25 outperform dense retrieval?

*Your answer here.*

### SA11. streaming implementation

Trace how `.stream()` propagates through `prompt | llm | StrOutputParser()`. Which components buffer and which pass through?

*Your answer here.*

### SA12. function calling schema

Describe the JSON schema format that LangChain's `@tool` decorator generates for OpenAI function calling.

*Your answer here.*

### SA13. cost tracking

How does the `CostTracker` in `class2_production_patterns` avoid incorrect cost calculation for local models?

*Your answer here.*

### SA14. retry middleware

Describe how `with_retry` works in LangChain and when you would use it in a production chain.

*Your answer here.*

### SA15. LangSmith eval run

Describe how to run an automated eval using LangSmith datasets. What is the output?

*Your answer here.*

### Part II Sample Answers

In [23]:
SA={"SA01": "Each `|` calls `__or__` on a `Runnable`, returning a `RunnableSequence`. Execution: (1) `PromptTemplate.invoke({'input': ...})` returns a `StringPromptValue`; (2) `llm.invoke(prompt_value)` returns an `AIMessage` or string; (3) `StrOutputParser().invoke(message)` extracts `.content` and returns a plain string. The full chain is lazy — nothing executes until `.invoke()`, `.stream()`, or `.batch()` is called on the sequence. `.stream()` delegates to each component's streaming implementation; components that don't support streaming return their result as a single chunk.", "SA02": "Buffer memory stores the full verbatim conversation — every message. It is simple and lossless but grows unbounded; context length limits are hit after many turns. Summary memory runs an LLM periodically to compress the conversation into a running summary, keeping context size bounded. Use buffer for short conversations (<10 turns) or when exact recall matters. Use summary for long-running assistants (customer support, coding sessions) where early context can be summarised without loss. Hybrid: ConversationSummaryBufferMemory keeps recent turns verbatim and summarises older ones.", "SA03": "`PydanticOutputParser` post-processes the LLM's free-form text by parsing JSON from it — if the model outputs invalid JSON or missing fields, parsing fails. It relies on the prompt to instruct the model to output JSON correctly. `with_structured_output` uses the provider's native mechanism (OpenAI function calling, Anthropic tool use, or grammar-constrained decoding) to constrain generation to valid JSON at the token level. The model physically cannot produce invalid JSON. For providers that don't support native structured output (HuggingFace local models), `with_structured_output` falls back to prompt-based parsing.", "SA04": "IndexFlatL2 computes exact L2 distances to all N stored vectors — O(N×D) per query. For N=10K and D=384, this is ~3.8M operations, taking ~5ms on CPU — acceptable. For N=1M it becomes ~500ms — unacceptable. IndexHNSWFlat builds a hierarchical navigable small world graph enabling approximate nearest-neighbour search in O(log N) average case. At N=1M, query time drops to ~1ms at >99% recall. Trade-off: HNSW requires more memory and build time. For the curriculum's projects (N<50K), IndexFlatL2 is fine; for a 10K-document knowledge base, either works.", "SA05": "Small chunks (128 tokens): high precision — retrieved chunks are tightly relevant to the query, less noise. But may split relevant information across boundaries, reducing faithfulness. Large chunks (512-1024 tokens): better context continuity, less splitting of related information. But retrieved chunks contain more irrelevant content (lower precision), increasing hallucination risk. Overlap (e.g. 10-20%): ensures sentences near chunk boundaries appear in at least two chunks, so edge-of-chunk information is always retrievable. Tradeoff: small chunks → better retrieval precision, worse context completeness. Large chunks → better context completeness, worse retrieval precision. Tune empirically via RAGAS context recall@k.", "SA06": "1) For each query in the golden set, run the full RAG pipeline: retrieve top-k docs, generate an answer. 2) Compute per-query: Faithfulness (are answer claims grounded in retrieved docs?), Context Recall (is the gold context in the retrieved docs?), Answer Relevance (is the answer relevant to the query?). 3) Aggregate to mean scores. 4) Compare against thresholds: Faithfulness>0.85, Context Recall>0.75, Answer Relevance>0.80. 5) Log failures (query, retrieved docs, answer) for manual inspection. In the curriculum, RAGAS runs as part of the eval harness — exit non-zero if any metric is below the configured `expected_band`.", "SA07": "Example: a chain that calls a web search tool, checks if the result contains an answer, and if not, reformulates the query and retries up to 3 times. In LCEL, the graph is a DAG — there is no way to loop back to an earlier node. You could unroll the loop (3 sequential search nodes) but the number of retries must be known at graph-construction time. LangGraph models this as a cycle: a `search_node` transitions to a `check_node`, which either routes to `END` (answer found) or back to `search_node` (retry needed). The cycle can run 0-N times depending on runtime conditions.", "SA08": "The `ToolRegistry` stores tools as a dict keyed by name. `registry.register(name, fn, args_schema)` adds a tool. `registry.get(name)` returns the tool metadata. `registry.dispatch(name, args_dict)` calls the underlying function with validated arguments. This decouples the agent loop from specific tool implementations — the ReAct parser only knows tool names and string inputs; dispatch handles type coercion and error catching. New tools are added by calling `register` without modifying the agent loop. The registry also exposes `registry.all()` to generate the tools description for the LLM prompt.", "SA09": "A LangSmith trace has a root span for the full chain invocation. Child spans: (1) `PromptTemplate.format` — captures the formatted prompt with retrieved context; (2) `LLM.generate` — captures input tokens, output tokens, model name, latency; (3) `Retriever.get_relevant_documents` — captures query and returned documents. Each span records: name, input, output, start/end time, status (success/error), and tags. In offline mode (`LANGSMITH_API_KEY` not set), the curriculum tracer writes these as JSONL: one JSON object per span, to `traces/langsmith_traces.jsonl`.", "SA10": "Hybrid retrieval runs BM25 keyword search and dense vector search in parallel, then fuses scores with reciprocal rank fusion (RRF) or linear interpolation. RRF: for each doc, score = sum_over_retrievers(1/(rank+k)). BM25 outperforms dense when: (1) the query contains rare proper nouns (product names, error codes) that are not in the dense model's vocabulary distribution — exact keyword match wins; (2) queries are very short (1-2 words) — not enough context for semantic embedding to activate; (3) domain vocabulary is highly specialised and out-of-distribution for the encoder. Dense outperforms for paraphrases and semantic similarity.", "SA11": "`PromptTemplate.stream()` yields the formatted prompt as a single chunk (no streaming within the template). `llm.stream()` yields individual tokens as they are generated by the LLM. `StrOutputParser.transform()` passes each chunk through immediately (applies `.content` extraction per chunk). The result is a generator that yields decoded token strings with sub-second latency for the first token. If any component doesn't implement `stream()`, the chain falls back to `transform()` which calls `invoke()` and wraps the result as a single chunk — losing streaming benefit.", "SA12": "The `@tool` decorator introspects the Python function's signature and docstring to generate: `{name: fn_name, description: docstring, parameters: {type: 'object', properties: {arg_name: {type: inferred_type, description: from_annotation}}, required: [mandatory_args]}}`. The schema conforms to OpenAI's function-calling format (also compatible with Anthropic tool use). The LLM receives this schema alongside the user message and generates a structured tool call JSON like `{name: fn_name, arguments: {arg: value}}` instead of free text. The agent framework parses this JSON and dispatches the tool.", "SA13": "Local models (SmolLM2-135M, 360M) have zero inference cost (no API call). The `PRICES` dict sets their input and output price per 1K tokens to 0.0. The tracker computes `cost = (input_tokens/1000) * prices['input'] + (output_tokens/1000) * prices['output']`. For local models this is always $0.00. This enables correct aggregate cost reporting when the pipeline sometimes falls back to a cloud model (e.g. gpt-3.5-turbo on low-confidence answers). The tracker also logs `model` name per call, making it trivial to audit which queries triggered expensive API calls.", "SA14": "`chain.with_retry(stop_after_attempt=3, wait_exponential_jitter=True)` wraps the chain with a retry decorator. On `RateLimitError` or `ServiceUnavailableError`, the chain sleeps with exponential backoff + jitter and retries. After `stop_after_attempt` failures, it raises the last exception. Use in production when: calling third-party APIs (OpenAI, Anthropic) that rate-limit; when a network timeout is expected occasionally; or when a vector store occasionally returns 503. Do not use for `ValidationError` (retrying a malformed prompt always fails) or business logic errors.", "SA15": "1) Create a dataset in LangSmith with your golden query-answer pairs using `langsmith.create_dataset()`. 2) Define an evaluator function that scores each output (e.g. `RougeEvaluator`). 3) Call `langsmith.evaluate(pipeline, data=dataset_name, evaluators=[evaluator], experiment_prefix='v2')`. LangSmith runs each example through the pipeline, records the output alongside the evaluator score, and aggregates into an experiment report. Output: a comparison table showing metric means per experiment — you can compare `v1` vs `v2` side-by-side in the UI. The curriculum's offline fallback records the same data to a local JSONL file for inspection without a LangSmith account."}
for k,v in SA.items(): print(f'\n── {k} ──\n{v}')


── SA01 ──
Each `|` calls `__or__` on a `Runnable`, returning a `RunnableSequence`. Execution: (1) `PromptTemplate.invoke({'input': ...})` returns a `StringPromptValue`; (2) `llm.invoke(prompt_value)` returns an `AIMessage` or string; (3) `StrOutputParser().invoke(message)` extracts `.content` and returns a plain string. The full chain is lazy — nothing executes until `.invoke()`, `.stream()`, or `.batch()` is called on the sequence. `.stream()` delegates to each component's streaming implementation; components that don't support streaming return their result as a single chunk.

── SA02 ──
Buffer memory stores the full verbatim conversation — every message. It is simple and lossless but grows unbounded; context length limits are hit after many turns. Summary memory runs an LLM periodically to compress the conversation into a running summary, keeping context size bounded. Use buffer for short conversations (<10 turns) or when exact recall matters. Use summary for long-running assistant

---
## Part III — Coding (10 × 3 pts)

### CC01. LCEL chain
Using `FakeLLM` and `FakeEmbed`, build `rag_chain(store, llm, k=2)` that takes `{'query': '...'}`, retrieves k docs, formats a prompt `'Context: {docs}\nQuery: {query}'`, calls llm, returns string.

In [24]:
def rag_chain(store, llm, k=2):
    def run(inputs):
        docs=store.search(inputs['query'],k)
        ctx=' '.join(docs)
        prompt=f'Context: {ctx}\nQuery: {inputs["query"]}'
        return llm.invoke(prompt)
    return run


In [25]:
try:
    store=FakeVectorStore()
    store.add(['LoRA is low-rank adaptation','QLoRA uses NF4 quantisation'])
    llm=FakeLLM(['answer'])
    chain=rag_chain(store,llm)
    result=chain({'query':'what is LoRA'})
    assert isinstance(result,str) and len(result)>0
    print('CC01 PASS:',result)
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC01 PASS: answer


### CC02. Streaming chain
Using `FakeLLM.stream()`, implement `stream_chain(llm, prompt_str)` that yields tokens from the LLM's stream output. Collect all tokens and verify the full text matches `llm.invoke(prompt_str)`.

In [26]:
def stream_chain(llm, prompt_str):
    for tok in llm.stream(prompt_str):
        yield tok


In [27]:
try:
    llm=FakeLLM(['hello world from LLM'])
    full=''.join(stream_chain(llm,'test prompt'))
    assert 'hello' in full and 'LLM' in full,full
    print('CC02 PASS:',repr(full))
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC02 PASS: 'hello world from LLM '


### CC03. Conversation buffer
Implement `ConvBuffer` with `add_turn(user, assistant)` and `format_history()` returning a string like `'User: ...\nAssistant: ...\n'` for all turns.

In [28]:
class ConvBuffer:
    def __init__(self): self.turns=[]
    def add_turn(self,user,assistant): self.turns.append((user,assistant))
    def format_history(self):
        pass


In [29]:
try:
    cb=ConvBuffer()
    cb.add_turn('What is LoRA?','LoRA is low-rank adaptation.')
    cb.add_turn('And QLoRA?','QLoRA adds quantisation.')
    h=cb.format_history()
    assert 'LoRA is low-rank' in h and 'QLoRA adds' in h
    print('CC03 PASS:',repr(h[:80]))
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

  ✗ Incomplete (TypeError: argument of type 'NoneType' is not iterable)


### CC04. Pydantic validator
Given `class ExtractionResult` with fields `entity: str` and `confidence: float` (0-1), write `parse_llm_output(text)` that parses a JSON string from `text` and returns an `ExtractionResult`. Return `None` on parse error.

In [30]:
import json
class ExtractionResult:
    def __init__(self,entity,confidence):
        assert 0<=confidence<=1
        self.entity=entity; self.confidence=confidence
    def __repr__(self): return f'ExtractionResult({self.entity},{self.confidence})'

def parse_llm_output(text):
    pass


In [31]:
try:
    good=parse_llm_output('{"entity":"LoRA","confidence":0.95}')
    assert good is not None and good.entity=='LoRA' and abs(good.confidence-0.95)<1e-9
    bad=parse_llm_output('not json at all')
    assert bad is None
    print('CC04 PASS:',good,bad)
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

  ✗ Incomplete (AssertionError: )


### CC05. ReAct parser
Write `parse_react(text)` returning `{'action':str|None,'action_input':str|None,'final_answer':str|None}`.

In [32]:
import re
def parse_react(text):
    fa=re.search(r'Final Answer[:\s]+(.+?)(?:\n|$)',text,re.I)
    act=re.search(r'Action[:\s]+(\w+)',text,re.I)
    ai=re.search(r'Action Input[:\s]+(.+?)(?:\n|$)',text,re.I)
    return {'final_answer':fa.group(1).strip() if fa else None,
            'action':act.group(1).strip() if act else None,
            'action_input':ai.group(1).strip() if ai else None}


In [33]:
try:
    r1=parse_react('Action: search\nAction Input: LoRA paper\n')
    assert r1['action']=='search' and r1['action_input']=='LoRA paper'
    r2=parse_react('Final Answer: LoRA uses rank-16 matrices.')
    assert r2['final_answer']=='LoRA uses rank-16 matrices.'
    print('CC05 PASS')
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC05 PASS


### CC06. In-memory vector store with MMR
Using `FakeVectorStore`, implement `mmr_search(store, query, k=3, lambda_=0.5)` that returns k documents balancing relevance (cosine to query) and diversity (min cosine to already-selected docs).

In [34]:
def mmr_search(store, query, k=3, lambda_=0.5):
    # Retrieve 2k candidates, then greedily select k with MMR
    emb=FakeEmbed()
    q_vec=emb.embed(query)
    candidates=store.search(query,min(2*k,len(store._docs)))
    selected=[]; remaining=list(candidates)
    for _ in range(min(k,len(remaining))):
        if not selected:
            # pick most relevant
            best=max(remaining,key=lambda d:cosine(emb.embed(d),q_vec))
        else:
            def mmr_score(d):
                rel=cosine(emb.embed(d),q_vec)
                red=max(cosine(emb.embed(d),emb.embed(s)) for s in selected)
                return lambda_*rel-(1-lambda_)*red
            best=max(remaining,key=mmr_score)
        selected.append(best); remaining.remove(best)
    return selected


In [35]:
try:
    store=FakeVectorStore()
    store.add(['LoRA low-rank adaptation','LoRA saves memory','QLoRA quantises base model','DPO preference optimisation'])
    results=mmr_search(store,'LoRA',k=2)
    assert len(results)==2
    print('CC06 PASS:',results)
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC06 PASS: ['LoRA low-rank adaptation', 'QLoRA quantises base model']


### CC07. LangSmith JSONL tracer
Implement `JSONLTracer` with `log_span(name, inputs, outputs, latency_ms)` that appends a JSON line to `traces/trace.jsonl`. Create the directory if needed.

In [36]:
import json, os, tempfile
class JSONLTracer:
    def __init__(self,path):
        os.makedirs(os.path.dirname(path),exist_ok=True)
        self.path=path
    def log_span(self,name,inputs,outputs,latency_ms):
        record={'name':name,'inputs':inputs,'outputs':outputs,'latency_ms':latency_ms}
        with open(self.path,'a') as f: f.write(json.dumps(record)+'\n')


In [37]:
try:
    import tempfile,json,os
    tmp=tempfile.mkdtemp()
    tr=JSONLTracer(os.path.join(tmp,'traces','trace.jsonl'))
    tr.log_span('LLM',{'prompt':'hello'},{'output':'world'},42)
    with open(tr.path) as f: line=json.loads(f.readline())
    assert line['name']=='LLM' and line['latency_ms']==42
    print('CC07 PASS:',line)
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC07 PASS: {'name': 'LLM', 'inputs': {'prompt': 'hello'}, 'outputs': {'output': 'world'}, 'latency_ms': 42}


### CC08. RAGAS faithfulness (mock)
Given a list of `claims` (strings extracted from an answer) and a list of `context_sentences`, implement `faithfulness_score(claims, context)` returning fraction of claims for which ANY context sentence has cosine similarity ≥ 0.7 to the claim.

In [38]:
def faithfulness_score(claims, context, threshold=0.7):
    emb=FakeEmbed()
    grounded=0
    for claim in claims:
        cv=emb.embed(claim)
        if any(cosine(cv,emb.embed(c))>=threshold for c in context):
            grounded+=1
    return grounded/len(claims) if claims else 1.0


In [39]:
try:
    claims=['LoRA uses low-rank matrices','LoRA saves VRAM']
    ctx=['LoRA inserts low-rank adapter matrices into attention layers','LLMs are large']
    score=faithfulness_score(claims,ctx)
    assert 0<=score<=1,'out of range'
    print(f'CC08 PASS: faithfulness={score:.2f}')
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC08 PASS: faithfulness=1.00


### CC09. Cost tracker with per-call log
Implement `CostTracker` with `record(model,in_tok,out_tok)` and `summary()` returning `{'total_cost':float,'calls':int,'by_model':{model:cost}}`.

In [40]:
PRICES={'SmolLM2-135M-Instruct':{'in':0.0,'out':0.0},'gpt-3.5-turbo':{'in':0.002,'out':0.002}}
class CostTracker:
    def __init__(self): self._log=[]
    def record(self,model,in_tok,out_tok):
        p=PRICES.get(model,{'in':0.0,'out':0.0})
        cost=(in_tok/1000)*p['in']+(out_tok/1000)*p['out']
        self._log.append({'model':model,'cost':cost})
    def summary(self):
        by_model={}
        for e in self._log:
            by_model[e['model']]=by_model.get(e['model'],0)+e['cost']
        return {'total_cost':sum(by_model.values()),'calls':len(self._log),'by_model':by_model}


In [41]:
try:
    ct=CostTracker()
    ct.record('gpt-3.5-turbo',500,100)
    ct.record('SmolLM2-135M-Instruct',1000,200)
    s=ct.summary()
    assert abs(s['total_cost']-0.0012)<1e-9,s['total_cost']
    assert s['calls']==2
    print('CC09 PASS:',s)
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC09 PASS: {'total_cost': 0.0012000000000000001, 'calls': 2, 'by_model': {'gpt-3.5-turbo': 0.0012000000000000001, 'SmolLM2-135M-Instruct': 0.0}}


### CC10. Conditional LangGraph router
Using plain Python (no langgraph import), implement `run_react_graph(task, llm, max_steps=5)` using a loop that calls the LLM, parses ReAct output with `parse_react`, calls a mock tool if action is found, and returns the final answer.

In [42]:
def run_react_graph(task, llm, tools=None, max_steps=5):
    tools=tools or {}
    history=''
    for _ in range(max_steps):
        prompt=f'Task: {task}\n{history}\nThought:'
        raw=llm.invoke(prompt)
        p=parse_react(raw)
        if p['final_answer']: return p['final_answer']
        if p['action']:
            fn=tools.get(p['action'],lambda x:'tool not found')
            obs=fn(p['action_input'])
            history+=f' {raw}\nObservation: {obs}\nThought:'
        else:
            return raw.strip()[:100]
    return 'max steps reached'


In [43]:
try:
    llm=FakeLLM(['Action: calc\nAction Input: 2+2','Final Answer: 4'])
    result=run_react_graph('What is 2+2?',llm,tools={'calc':lambda x:str(eval(x))})
    assert result=='4',result
    print('CC10 PASS:',result)
except (AssertionError,TypeError,AttributeError,NotImplementedError) as _e:
    print(f'  ✗ Incomplete ({type(_e).__name__}: {_e})')

CC10 PASS: 4


---
## Part IV — Case Studies (5 × 4 pts)

### CS01. LangGraph infinite loop (Anthropic engineering)

Your LangGraph ReAct agent loops forever — `max_steps` is never hit. Logs show the same tool call repeated 50 times. Diagnose and fix.

*Your answer here.*

### CS02. RAGAS faithfulness collapse (OpenAI eval)

Your RAG pipeline goes from faithfulness=0.88 to 0.41 after switching from top-3 to top-1 retrieval to reduce latency. Explain and propose a fix that preserves latency.

*Your answer here.*

### CS03. LangSmith missing spans (Meta production debug)

Your LangSmith traces show the root span but no child spans for retrieval or LLM calls. Diagnose the most likely cause.

*Your answer here.*

### CS04. Semantic cache coherence (Amazon production)

After a model upgrade (SmolLM2-135M → 360M), the semantic cache is still serving responses generated by the 135M model. Users report answers are outdated/different quality. Fix.

*Your answer here.*

### CS05. Multi-query high latency (Google ML onsite)

Multi-query retrieval generates 3 queries and runs 3 FAISS searches. This triples latency. Redesign to maintain quality while reducing latency.

*Your answer here.*

### Part IV Rubric

In [44]:
R={"CS01": ["Root cause: the tool is returning the same observation every time ('No results found'), the LLM continues to reformulate and retry the same action without progress, and there's no deduplication check.", "Fix 1: track visited (action, action_input) pairs; if the same pair appears twice in one session, break the loop with 'Stuck in loop' final answer.", "Fix 2: inspect the LangGraph state — check if `done=True` routing condition is correct. A bug where `done` is never set to `True` means the graph never reaches `END`.", "Fix 3: add a step counter to the state dict and route to END when steps > max_steps. Verify the conditional edge condition: `if state['steps'] >= max_steps: return END`.", "Fix 4: add a tool result cache — if the same tool+input was called in this session, return the cached result instead of calling the tool again."], "CS02": ["With top-1 retrieval, only one document is available to ground the answer. If that document doesn't contain all the facts needed for a complete answer, the LLM invents additional claims — causing faithfulness collapse.", "Fix 1: keep top-3 retrieval but use contextual compression — extract only the relevant sentences from each doc before feeding to the LLM. Context size stays small (like top-1) but more information is available.", "Fix 2: use an extractive rather than generative answer: force the LLM to quote directly from the retrieved passage (`'Answer ONLY using exact quotes from the context'`). Lower faithfulness risk but less natural answers.", "Fix 3: switch to smaller, faster chunks (128 tokens instead of 512) and keep k=3. Each chunk is smaller so total context size with k=3 is the same as k=1 with large chunks, but recall is higher."], "CS03": ["Most likely: the chain is constructed without proper tracing callbacks. LangSmith tracing requires either LANGCHAIN_TRACING_V2=true env var or explicit `RunnableConfig(callbacks=[LangSmithCallbackHandler()])` passed to `.invoke()`.", "Check: is `LANGSMITH_API_KEY` set in the environment? Without it, the tracer is in offline mode or disabled entirely.", "Check: is the chain using `.invoke()` or `.run()`? Old `run()` API may not propagate callbacks correctly through all components.", "Check: if using a custom `HuggingFacePipeline` wrapper, it may not call LangChain's callback hooks. Verify the wrapper calls `self.callbacks.on_llm_start` and `on_llm_end`.", "Fix: explicitly pass `config=RunnableConfig(callbacks=[tracer])` to all `.invoke()` calls, or set `LANGCHAIN_TRACING_V2=true` globally."], "CS04": ["The cache stores responses keyed by embedding similarity. After a model upgrade, old cached responses are from the weaker model — the cache is stale.", "Fix 1: flush the cache on every model version change. Add a `model_version` field to cached entries; reject cache hits where the version doesn't match the current model.", "Fix 2: include model version in the cache key (e.g. hash of model name + embedding). Old entries are automatically unreachable after the model changes.", "Fix 3: use TTL (time-to-live) on cache entries — entries expire after 24h. Old responses naturally age out.", "Process: model version changes should always trigger cache invalidation as part of the deploy checklist. Add a `POST /cache/flush` admin endpoint to Cloud Run services."], "CS05": ["Option 1: run the 3 FAISS searches in parallel (asyncio or threading). Latency drops from 3× to ~1.1× (shared I/O overhead). For CPU-bound FAISS, use a thread pool with 3 workers.", "Option 2: generate queries sequentially but search only if the previous search returns < k/2 relevant results. Early termination when recall is already sufficient.", "Option 3: reduce from 3 reformulations to 2, or use a single query but with a larger k (k=9 instead of k=3). Union and deduplicate results. Often matches recall of multi-query at 1/3 the cost.", "Option 4: cache multi-query results at the full result-set level (not just per-query). If the same original question was asked before, return the cached union of results.", "Recommendation: Option 1 (parallel searches) is the best tradeoff — full quality of multi-query at near-single-query latency."]}
for k,pts in R.items():
    print(f'\n── {k} ──')
    for i,p in enumerate(pts,1): print(f'  {i}. {p}')


── CS01 ──
  1. Root cause: the tool is returning the same observation every time ('No results found'), the LLM continues to reformulate and retry the same action without progress, and there's no deduplication check.
  2. Fix 1: track visited (action, action_input) pairs; if the same pair appears twice in one session, break the loop with 'Stuck in loop' final answer.
  3. Fix 2: inspect the LangGraph state — check if `done=True` routing condition is correct. A bug where `done` is never set to `True` means the graph never reaches `END`.
  4. Fix 3: add a step counter to the state dict and route to END when steps > max_steps. Verify the conditional edge condition: `if state['steps'] >= max_steps: return END`.
  5. Fix 4: add a tool result cache — if the same tool+input was called in this session, return the cached result instead of calling the tool again.

── CS02 ──
  1. With top-1 retrieval, only one document is available to ground the answer. If that document doesn't contain all the 